# Advanced Distribution and Multivariate Plots

This notebook covers chart types designed to reveal **how data is distributed**
— not just what value a category has, but the full shape of the data.
The charts progress from one variable to two to many:

| Family | What it answers |
|---|---|
| Count / Histogram / KDE | How is one variable distributed? |
| Strip / Swarm / Box / Violin | How does that distribution differ across groups? |
| Jointplot | What is the joint distribution of two variables? |
| Pairplot / Heatmap | How do all variable pairs relate to each other? |

**Dataset:** Palmer Penguins — 333 observations, 3 species, 4 body measurements.
11 rows with missing values are dropped upfront so all charts use the same clean data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

penguins = sns.load_dataset('penguins').dropna()

## Count Plot — 1D categorical frequency

A count plot is a bar chart where matplotlib counts the observations for you.
`sns.countplot()` accepts a categorical column and draws one bar per category.
Adding `hue` splits each bar by a second category (here: sex), making it easy
to compare composition across groups.
`order` pins the bar sequence; without it, seaborn uses the order rows appear in the data.

In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(
    data=penguins,
    x='species',
    hue='sex',
    order=['Adelie', 'Chinstrap', 'Gentoo'],
)
plt.title('Penguin count by species and sex')
plt.xlabel('Species')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## Histogram — 1D continuous distribution

A histogram bins a continuous variable and draws a bar for each bin whose
height is the number of observations in that bin. The choice of `bins` controls
the resolution: too few bins hide structure, too many create noise.

`sns.histplot()` with `hue` overlays one histogram per group using transparency.
`multiple='stack'` or `multiple='dodge'` are alternatives when overlap is a problem.

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(
    data=penguins,
    x='flipper_length_mm',
    hue='species',
    bins=25,
    alpha=0.7,
)
plt.title('Flipper length distribution by species')
plt.xlabel('Flipper length (mm)')
plt.tight_layout()
plt.show()

## KDE Plot — smoothed continuous distribution

A **Kernel Density Estimate** replaces the discrete bins of a histogram with a
smooth curve by placing a small Gaussian bell ("kernel") on every data point
and summing them. The result represents an estimate of the true probability
density function.

`bw_adjust` scales the bandwidth (smoothing amount): values < 1 show more
detail, values > 1 produce a smoother curve. `fill=True` shades under the
curve, making overlapping species easier to distinguish.

In [ ]:
plt.figure(figsize=(8, 4))
sns.kdeplot(
    data=penguins,
    x='flipper_length_mm',
    hue='species',
    fill=True,
    bw_adjust=0.8,
    alpha=0.4,
)
plt.title('Flipper length KDE by species')
plt.xlabel('Flipper length (mm)')
plt.tight_layout()
plt.show()

## Strip Plot — raw data across categories

A strip plot places every individual observation as a dot along a categorical
axis. It is the most honest chart in this family because nothing is hidden or
aggregated — you see exactly how many points there are and where they fall.

`jitter=True` adds a small random horizontal offset so overlapping points
become visible. With many observations the overlap becomes severe; swarm plots
solve this systematically.

In [ ]:
plt.figure(figsize=(7, 5))
sns.stripplot(
    data=penguins,
    x='species',
    y='body_mass_g',
    hue='sex',
    jitter=True,
    dodge=True,
    alpha=0.7,
    size=4,
)
plt.title('Body mass by species — strip plot')
plt.xlabel('Species')
plt.ylabel('Body mass (g)')
plt.tight_layout()
plt.show()

## Swarm Plot — non-overlapping raw data

A swarm plot solves strip plot overlap by placing each point using a
deterministic algorithm that shifts dots sideways just enough to avoid
collisions. The horizontal spread of the swarm therefore encodes density:
a wider swarm means more observations at that value.

This makes the swarm plot a useful middle ground: all data is shown (unlike
box plots) and points do not pile up (unlike strip plots). The trade-off is
that it scales poorly beyond a few hundred points per group.

In [ ]:
plt.figure(figsize=(7, 5))
sns.swarmplot(
    data=penguins,
    x='species',
    y='body_mass_g',
    hue='sex',
    dodge=True,
    size=3.5,
)
plt.title('Body mass by species — swarm plot')
plt.xlabel('Species')
plt.ylabel('Body mass (g)')
plt.tight_layout()
plt.show()

## Box Plot — five-number statistical summary

A box plot condenses each group's distribution into five statistics:
- **Box**: 25th percentile (Q1) to 75th percentile (Q3) — the interquartile range (IQR)
- **Line inside box**: median (Q2)
- **Whiskers**: extend to the last data point within 1.5 × IQR from the box edges
- **Dots beyond whiskers**: statistical outliers

Box plots are compact and easy to compare across many groups, but they hide
the distribution shape — a bimodal distribution looks identical to a uniform one.

In [ ]:
plt.figure(figsize=(7, 5))
sns.boxplot(
    data=penguins,
    x='species',
    y='body_mass_g',
    hue='sex',
)
plt.title('Body mass by species — box plot')
plt.xlabel('Species')
plt.ylabel('Body mass (g)')
plt.tight_layout()
plt.show()

### Manipulating whiskers

The `whis` parameter controls how far whiskers extend from the box edges.
Its default value of `1.5` means: reach to the furthest data point that still
falls within **1.5 × IQR** of Q1 or Q3. Any point beyond that is an outlier.

- **Lower `whis`** → shorter whiskers → more points classified as outliers
- **Higher `whis`** → longer whiskers → fewer points classified as outliers
- **`whis=[p_low, p_high]`** → whisker tips land exactly at those percentiles,
  bypassing the IQR rule entirely (useful when you have a target coverage, e.g. 5–95 %)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 5), sharey=True)

for ax, whis, label in zip(
    axes,
    [0.5, 1.5, [5, 95]],
    ['whis=0.5  (strict)', 'whis=1.5  (default)', 'whis=[5, 95]  (percentile)'],
):
    sns.boxplot(data=penguins, x='species', y='body_mass_g', whis=whis, ax=ax)
    ax.set_title(label)
    ax.set_xlabel('Species')
    ax.set_ylabel('Body mass (g)' if ax is axes[0] else '')

fig.suptitle('Effect of whisker length on outlier detection')
plt.tight_layout()
plt.show()

### Manipulating outliers

By default, outliers are drawn as small diamonds beyond the whisker tips.
Two common adjustments:

- **`showfliers=False`** — hide outliers entirely. Useful when they distract from
  the main comparison, but risks concealing important extreme values.
- **`flierprops`** — pass a dict of matplotlib marker properties to restyle
  outliers: change shape (`marker`), colour, size, or transparency.
  Making them large and red turns outliers into a signal rather than noise.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5), sharey=True)

sns.boxplot(data=penguins, x='species', y='body_mass_g',
            showfliers=False, ax=ax1)
ax1.set_title('Outliers hidden  (showfliers=False)')
ax1.set_xlabel('Species')
ax1.set_ylabel('Body mass (g)')

sns.boxplot(
    data=penguins, x='species', y='body_mass_g',
    flierprops=dict(marker='D', markerfacecolor='tomato',
                    markeredgecolor='tomato', markersize=7, alpha=0.7),
    ax=ax2,
)
ax2.set_title('Custom outlier markers  (flierprops)')
ax2.set_xlabel('Species')
ax2.set_ylabel('')

plt.tight_layout()
plt.show()

### Time series box plot — seasonal patterns

When the x-axis is a time unit (month, quarter, year), a box plot per period
shows how **the full distribution shifts over time**, not just the mean.
This is far more informative than a line chart of averages for seasonal data.

The `flights` dataset (built into seaborn) records monthly airline passenger
counts from 1949 to 1960. Each box summarises 12 annual measurements for
that month, making summer peaks and winter troughs immediately visible —
and showing that variance grows alongside the trend.

In [ ]:
flights = sns.load_dataset('flights')
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

plt.figure(figsize=(11, 5))
sns.boxplot(data=flights, x='month', y='passengers', order=month_order)
plt.title('Monthly airline passengers 1949–1960 — seasonal pattern')
plt.xlabel('Month')
plt.ylabel('Passengers')
plt.tight_layout()
plt.show()

## Violin Plot — box plot + KDE

A violin plot combines the statistical summary of a box plot with the shape
information of a KDE. The width of the "violin" at any height is proportional
to the estimated density there — so a bulge means many observations at that value.

`inner='box'` draws a miniature box plot inside the violin.
`split=True` (requires `hue` with exactly two levels) mirrors the two groups
on opposite sides of the violin, making the male/female comparison very direct.

In [ ]:
plt.figure(figsize=(7, 5))
sns.violinplot(
    data=penguins,
    x='species',
    y='body_mass_g',
    hue='sex',
    split=True,
    inner='box',
)
plt.title('Body mass by species — violin plot')
plt.xlabel('Species')
plt.ylabel('Body mass (g)')
plt.tight_layout()
plt.show()

### Box plot vs Violin plot — why shape matters

The two charts below use **exactly the same data**: body mass of all 333 penguins,
with no grouping by species.

The box plot reports a valid five-number summary, but it suggests a roughly
symmetric, unimodal distribution. The violin immediately reveals the truth:
the data is **bimodal** — there is a lighter cluster (Adelie + Chinstrap,
~3 500 g) and a heavier cluster (Gentoo, ~5 000 g). The box plot is not
wrong, but it is incomplete.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 5), sharey=True)

sns.boxplot(data=penguins, y='body_mass_g', color='steelblue', ax=ax1)
ax1.set_title('Box plot')
ax1.set_ylabel('Body mass (g)')
ax1.set_xlabel('')

sns.violinplot(data=penguins, y='body_mass_g', color='steelblue', inner='box', ax=ax2)
ax2.set_title('Violin plot')
ax2.set_ylabel('')
ax2.set_xlabel('')

fig.suptitle('All penguins — body mass (no species split)\nThe violin reveals a bimodal shape invisible in the box plot')
plt.tight_layout()
plt.show()

## Jointplot — 2D scatter with marginal distributions

A jointplot combines a central 2D chart with marginal histograms (or KDEs) on
the top and right edges, showing each variable's individual distribution
alongside the joint relationship.

The `hue` parameter colours points by species and draws one KDE per species
on the marginals, turning the plot into a three-way comparison. `kind` can be
changed to `'kde'` or `'hex'` for density-focused alternatives (incompatible
with `hue`).

In [ ]:
g = sns.jointplot(
    data=penguins,
    x='bill_length_mm',
    y='flipper_length_mm',
    hue='species',
    height=6,
    alpha=0.7,
)
g.set_axis_labels('Bill length (mm)', 'Flipper length (mm)')
g.figure.suptitle('Bill length vs flipper length by species', y=1.01)
plt.tight_layout()
plt.show()

## Pairplot — all pairwise relationships at once

A pairplot arranges every pair of numeric variables in a grid. The diagonal
shows each variable's marginal distribution (KDE by default when `hue` is set);
the off-diagonal cells show scatter plots of each pair.

With `hue='species'` each species gets a colour, making clusters and separations
immediately visible across all variable combinations in one view.
Note that pairplots are computationally heavier than single charts — they
generate n² subplots where n is the number of numeric variables.

In [ ]:
g = sns.pairplot(
    data=penguins,
    hue='species',
    diag_kind='kde',
    plot_kws={'alpha': 0.5, 'edgecolors': 'none', 's': 20},
)
g.figure.suptitle('Pairplot — all numeric variables by species', y=1.01)
plt.show()

## Correlation Heatmap — multivariate mathematical summary

**Correlation** measures the strength and direction of the linear relationship
between two variables. When one variable tends to increase as the other
increases, they are *positively* correlated; when one increases as the other
decreases, they are *negatively* correlated; when they move independently,
the correlation is near zero.

The most common measure is the **Pearson correlation coefficient** (*r*),
which always falls in the range [−1, +1]:

| Value | Meaning |
|---|---|
| +1.0 | Perfect positive linear relationship |
| +0.7 to +0.9 | Strong positive |
| +0.4 to +0.6 | Moderate positive |
| ~0 | No linear relationship |
| −0.4 to −0.6 | Moderate negative |
| −1.0 | Perfect negative linear relationship |

For a full mathematical treatment see:
https://en.wikipedia.org/wiki/Correlation

A correlation heatmap computes *r* for every pair of numeric variables and
displays the resulting matrix as a grid of coloured squares.

`annot=True` prints the coefficient inside each cell. `cmap='coolwarm'`
uses blue for negative and red for positive — intuitive and colourblind-friendly.
The diagonal is always 1.0 (a variable is perfectly correlated with itself).

A heatmap is a mathematical summary, not a distribution chart — it cannot
distinguish linear from non-linear relationships, and it is blind to outliers.
Use it as a first-pass screening tool, then follow up with scatter/jointplots.

In [ ]:
corr = penguins.select_dtypes(include='number').corr()

plt.figure(figsize=(6, 5))
sns.heatmap(
    corr,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
)
plt.title('Pearson correlation — penguin measurements')
plt.tight_layout()
plt.show()